# EMG2Pose Challenge: Introduction

This notebook gives a basic introduction to Meta's open-access [emg2pose dataset](https://github.com/facebookresearch/emg2pose/tree/main) of Surface electromyography (sEMG) recordings paired with ground-truth, motion-capture recordings of the hand gestures. This introduction is based on their [getting_started.ipynb](https://github.com/facebookresearch/emg2pose/blob/main/notebooks/getting_started.ipynb) notebook, which contains nice visualizations that are not included here. We encourage you to try out their original notebook on your local machine using their [setup instructions](https://github.com/facebookresearch/emg2pose/tree/main#setup) to gain a understanding of the dataset. 

This is intended to give you introductory information on how to load the dataset and set up a simple model. 

**Prerequisites**

Do this in a terminal window, before you run any notebook cells.

```bash
git clone https://github.com/facebookresearch/emg2pose.git
pip install -e emg2pose/
pip install h5py==3.11.0 hydra-core==1.3.2 omegaconf joblib==1.4.2 tqdm
```
Restart the notebook kernel and you are good to go! Note you need to rerun the pip commands whenever you start a new server.


## 1. Data location

This notebook uses a sample 600 MB mini dataset 

```bash
/hack-data/emg2pose/emg2pose_dataset_mini
```

Challenge training and eval datasets (~200GB) are located in: 

```bash
/hack-data/emg2pose/emg2pose_dataset_full/train/
/hack-data/emg2pose/emg2pose_dataset_full/eval/
```



In [ ]:
import glob
import os
from pathlib import Path

DATA_ROOT = Path.home() / "hack-data" / "emg2pose"

#This notebook: mini dataset
MINI_DIR = DATA_ROOT / "emg2pose_dataset_mini"

# Real training: use this in your own notebooks/code!
# TRAIN_DIR = DATA_ROOT / "emg2pose_dataset_full/train"
# EVAL_DIR  = DATA_ROOT / "emg2pose_dataset_full/eval"

sessions = sorted(glob.glob(os.path.join(MINI_DIR, "*.hdf5")))
print(f"Found {len(sessions)} session files under {MINI_DIR}")

## 2. Inspect the raw data

The dataset is made up of hdf5 files. Each `.hdf5` file is one hand, one stage, one session: time-aligned 16-channel sEMG at 2kHz and hand joint angles. `data.metadata` includes the per-file attributes (`user`, `session`, `stage`, `side`, `sample_rate`, ...)

In [ ]:
#load the data and print what is in it
from emg2pose.data import Emg2PoseSessionData

session_path = sessions[0]
data = Emg2PoseSessionData(hdf5_path=session_path)

print("fields:", list(data.fields))
print("metadata:", data.metadata)
print()
print(f"{'emg shape:':<20} {data['emg'].shape}")
print(f"{'joint_angles shape:':<20} {data['joint_angles'].shape}")
print(f"{'time shape:':<20} {data['time'].shape}")
print(f"{'# samples:':<20} {len(data)}")

In [ ]:
#make some basic visualizations of the data: 
#plot the sEMG readings and joint angles vs time
import matplotlib.pyplot as plt

window = data[0:2000]  # first 1 second at 2kHz

fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axs[0].plot(window["emg"])
axs[0].set_title(f"sEMG — {data.metadata.get('stage', '?')} ({data.metadata.get('side', '?')} hand)")
axs[0].set_ylabel("channel amplitude")

axs[1].plot(window["joint_angles"])
axs[1].set_title("Joint angles")
axs[1].set_ylabel("angle")
axs[1].set_xlabel("sample (2kHz)")

plt.tight_layout()
plt.show()

## Sample task: windowed regression

Predict the joint angle vector at time `t` given a window of sEMG readings leading up to and including time `t`.

**Input:** a fixed-length, causal window of raw sEMG (samples up to and including time `t` — no future context, since that's what a real-time wrist-worn deployment would actually have access to).

**Output:** the joint angle vector at time `t`.

A `no_ik_failure` mask is available per session — some timesteps have no valid ground truth because motion-capture markers were occluded (~12.7% of frames in the full dataset, per the paper). We skip any window whose target timestep falls in a failure region, otherwise you'd be training against garbage labels some fraction of the time.

## 3. Load the dataset for training

Put the raw data into a form PyTorch can read, and create the train/test sets.

In [ ]:
# Make a PyTorch Dataset that turns raw session files into training pairs (EMG window, target joint angle) 
# A standard PyTorch data-loading interface
import numpy as np
import torch
from torch.utils.data import Dataset


class EMGWindowDataset(Dataset):
    """Causal windowed regression: given sEMG samples [t - window_len + 1, t],
    predict the joint angle vector at time t.

    x: (window_len, 16) float32
    y: (num_joint_angles,) float32
    """

    def __init__(self, session_paths, window_len=250, stride=50):
        self.window_len = window_len
        self.stride = stride
        self.sessions = []
        self.index = []  # (session_idx, window_start)

        for path in session_paths:
            sd = Emg2PoseSessionData(hdf5_path=path)
            valid = sd.no_ik_failure  # bool mask, True = usable ground truth
            n = len(sd)
            session_idx = len(self.sessions)
            self.sessions.append(sd)
            for start in range(0, n - window_len, stride):
                target_t = start + window_len - 1  # causal: predict at the window's last sample
                if valid[target_t]:
                    self.index.append((session_idx, start))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        session_idx, start = self.index[i]
        sd = self.sessions[session_idx]
        end = start + self.window_len
        chunk = sd[start:end]

        x = torch.tensor(chunk["emg"], dtype=torch.float32)              # (window_len, 16)
        y = torch.tensor(chunk["joint_angles"][-1], dtype=torch.float32)  # (num_joints,)
        return x, y

In [ ]:
# Now load the data
from torch.utils.data import DataLoader

ds = EMGWindowDataset(sessions[:5], window_len=250, stride=50)
print(f"{len(ds)} windows from {len(sessions[:5])} sessions")

loader = DataLoader(ds, batch_size=32, shuffle=True)
x_batch, y_batch = next(iter(loader))
print("x_batch:", x_batch.shape)  # (batch, window_len, 16)
print("y_batch:", y_batch.shape)  # (batch, num_joints)

In [ ]:
# File-level split (stage-level, still all one user/session in the mini set)
train_files, eval_files = sessions[:-1], sessions[-1:]

train_ds = EMGWindowDataset(train_files, window_len=250, stride=50)
eval_ds = EMGWindowDataset(eval_files, window_len=250, stride=50)
print(f"train: {len(train_ds)} windows from {len(train_files)} files")
print(f"eval:  {len(eval_ds)} windows from {len(eval_files)} files")

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
eval_loader = DataLoader(eval_ds, batch_size=64, shuffle=False)

## 4. Define a metric and a baseline

Let's use mean absolute error as the loss function (MAE). Note the joint angles are stored as radians, so they must be converted to degrees. Also define a constant baseline: a model that always predicts the mean joint-angle vector. 

In [ ]:
# define MAE (degrees)
def mean_abs_error(loader, predict_fn):
    total_abs_err, total_count = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            pred = predict_fn(x)
            total_abs_err += torch.rad2deg((pred - y).abs()).sum().item()
            total_count += y.numel()
    return total_abs_err / total_count

In [ ]:
# define baseline: always predict the mean training joint-angle vector
train_targets = torch.stack([y for _, y in train_ds])
mean_angles = train_targets.mean(dim=0)

baseline_mae = mean_abs_error(eval_loader, predict_fn=lambda x: mean_angles.expand(x.shape[0], -1))
print(f"Constant-baseline eval MAE: {baseline_mae:.2f} degrees")

## 5. Build a minimal trainable model

This is just a basic little MLP model that you can replace with your own. It is recommended to normalize the input as is done in this example.

In [ ]:
# input normalization
# Per-channel EMG mean/std from the raw training signal (not the overlapping windows,
# to avoid over-weighting whatever region the stride happens to oversample).
train_emg = np.concatenate(
    [Emg2PoseSessionData(hdf5_path=p)[:]["emg"] for p in train_files], axis=0
)
emg_mean = torch.tensor(train_emg.mean(axis=0), dtype=torch.float32)
emg_std = torch.tensor(train_emg.std(axis=0), dtype=torch.float32) + 1e-6

def normalize(x):
    return (x - emg_mean) / emg_std

In [ ]:
# train a basic network to predict joint angles from sEMG windows
# output the MAE in degrees to compare to baseline
import math
import torch.nn as nn

class TinyMLP(nn.Module):
    def __init__(self, window_len=250, n_channels=16, n_joints=20, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(window_len * n_channels, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_joints),
        )

    def forward(self, x):
        return self.net(x)


model = TinyMLP()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.L1Loss()  # matches emg2pose's own AngleMAE metric

for epoch in range(5):
    model.train()
    train_loss_sum, train_count = 0.0, 0
    for x, y in train_loader:
        opt.zero_grad()
        pred = model(normalize(x))
        loss = loss_fn(pred, y)
        loss.backward()
        opt.step()
        train_loss_sum += loss.item() * x.shape[0]
        train_count += x.shape[0]

    model.eval()
    eval_mae = mean_abs_error(eval_loader, predict_fn=lambda x: model(normalize(x)))
    train_loss_rad = train_loss_sum / train_count  # L1Loss is on raw radians
    train_loss_deg = math.degrees(train_loss_rad)
    print(f"epoch {epoch+1}: train loss = {train_loss_rad:.4f} rad ({train_loss_deg:.2f} deg)  "
          f"eval MAE = {eval_mae:.2f} deg  (baseline = {baseline_mae:.2f} deg)")

There you have it: MAE here is how far off your predictions are on average from the actual joint angles your model is trying to predict, output here as degrees (the same unit as the thing being predicted). A perfect prediction would mean a MAE of 0 degrees. The baseline is the average joint angle for the whole dataset. You likely notice that the training loss is doing a little better than baseline, but the loss on the eval dataset is doing worse. A challenge with this dataset is generalizing beyond the training dataset, especially to data collected by other individuals (unlike this mini set, which is readings from just one person!)

## 6. Save the model

Save your model weights so you could use it later: for example copying it onto your server for testing FPGA resource utilization ;)

In [ ]:
#save your model weights
torch.save(model.state_dict(), "tiny_mlp_weights.pt")

You can then download this by right/ctrl clicking on it in the file viewer and choosing "Download"

To read it in a different notebook, you would also need to include the same model class definition you used in training (in this case your `TinyMLP` class). You can save it as a small .py file and copy that over too, or you can paste the same class definition into a cell in another notebook:

```bash
import torch
import torch.nn as nn

class TinyMLP(nn.Module):
    def __init__(self, window_len=250, n_channels=16, n_joints=20, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(window_len * n_channels, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_joints),
        )
    def forward(self, x):
        return self.net(x)

model = TinyMLP()
model.load_state_dict(torch.load("tiny_mlp_weights.pt"))
model.eval()
```

and then load the model like:

```bash
model = TinyMLP()
model.load_state_dict(torch.load("tiny_mlp_weights.pt"))
model.eval()
```
